In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
import re

In [0]:
tables = spark.catalog.listTables("spotify_project.bronze")
valid_table_names = [match for table in tables for match in re.findall(r"^streaming_history_audio_.*", table.name)]

COLUMN_RENAME_MAP = {
    'master_metadata_album_album_name': 'album_name',
    'master_metadata_album_artist_name': 'artist_name',
    'master_metadata_track_name': 'track_name',
    'ms_played': 'play_time',
    'ts': 'date_played'
}

COLUMN_KEEP = [
    'album_name',
    'artist_name',
    'track_name',
    'play_time',
    'date_played',
    'platform',
    'reason_end',
    'reason_start',
    'shuffle',
    'skipped'
]

COLUMN_TYPES = {
    'album_name': 'string',
    'artist_name': 'string',
    'track_name': 'string',
    'platform': 'string',
    'reason_end': 'string',
    'reason_start': 'string',
    'shuffle': 'boolean',
    'skipped': 'boolean',
    'date_played': 'date',
    'play_time': 'int',
}

for table_name in valid_table_names:
    # Extract year from table name: streaming_history_audio_{year}[_{num}]
    year = table_name.replace("streaming_history_audio_", "").split("_")[0]

    df = spark.table(f"spotify_project.bronze.{table_name}")

    # Drop columns that are entirely null
    null_counts = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).collect()[0]
    total_rows = df.count()
    non_null_cols = [c for c in df.columns if null_counts[c] < total_rows]
    df = df.select(*non_null_cols)

    # Rename columns based on the map
    for old_name, new_name in COLUMN_RENAME_MAP.items():
        if old_name in df.columns:
            df = df.withColumnRenamed(old_name, new_name)

    # Keep only columns in COLUMN_KEEP that exist
    keep_cols = [c for c in COLUMN_KEEP if c in df.columns]
    df = df.select(*keep_cols)

    # Cast datatypes
    for col_name, col_type in COLUMN_TYPES.items():
        if col_name in df.columns:
            if col_type == 'date':
                df = df.withColumn(col_name, F.to_date(F.col(col_name)))
            else:
                df = df.withColumn(col_name, F.col(col_name).cast(col_type))

    # Coalesce play_time to 0
    if 'play_time' in df.columns:
        df = df.withColumn('play_time', F.coalesce(F.col('play_time'), F.lit(0)))

    # Drop duplicate rows
    df = df.dropDuplicates()

    # Drop rows where BOTH artist_name and track_name are null OR all columns are null
    all_null_condition = F.lit(True)
    for c in df.columns:
        all_null_condition = all_null_condition & F.col(c).isNull()
    both_null_condition = F.col('artist_name').isNull() & F.col('track_name').isNull()
    df = df.filter(~both_null_condition & ~all_null_condition)

    silver_table = f"spotify_project.silver.streaming_history_{year}"

    if spark.catalog.tableExists(silver_table):
        # Table already exists for this year — append the data
        df.write.mode("append").saveAsTable(silver_table)
    else:
        # First table for this year — create it
        df.write.mode("overwrite").saveAsTable(silver_table)




In [0]:
%sql
-- SELECT SUM(counts) FROM (
--     SELECT COUNT(*) AS counts FROM spotify_project.bronze.streaming_history_audio_2018 UNION
--     SELECT COUNT(*) AS counts FROM spotify_project.bronze.streaming_history_audio_2018_1 UNION
--     SELECT COUNT(*) AS counts FROM spotify_project.bronze.streaming_history_audio_2018_2
-- )

SELECT COUNT(*) FROM spotify_project.silver.streaming_history_2018